In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd().resolve().parent
DATA_PATH = BASE_DIR / "data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(DATA_PATH)

In [8]:
features = df.copy()

# Đưa các giá trị về ln(1+x) để giảm độ chênh lệch dữ liệu
features["log_amount"] = np.log1p(features["amount"])

# trong mô tả của dataset, 1 step là 1 giờ 
features["hour_of_day"] = (
    (features["step"] - 1) % 24
) #Chỗ này trừ 1 rồi modulo cho 24 để lấy ra giờ của ngày đó

features["day_index"] = (
    (features["step"] - 1) // 24
) #Chỗ này trừ 1 rồi chia nguyên cho 24 để lấy ngày 


features["orig_balance_ratio"] = (
    features["amount"] / #Amount chia cho oldbalanceOrg để lấy ra được tỷ lệ so với tiền hiện tại
    (features["oldbalanceOrg"].abs() + 1) # Cộng 1 để tránh chia cho 0
)

features["orig_insufficient_balance"] = ( # Kiểm tra xem thử tiền hiện tại có lớn hơn oldbalance ko?
    features["amount"] > features["oldbalanceOrg"]
).astype(int)

display(features.head())

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,log_amount,hour_of_day,day_index,orig_balance_ratio,orig_insufficient_balance
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,9.194276,0,0,0.057834,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,7.531166,0,0,0.087731,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,5.204007,0,0,0.994505,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,5.204007,0,0,0.994505,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,9.364703,0,0,0.280788,0


In [10]:
#Đưa vào danh sách chọn nè
feature_columns = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "log_amount",
    "hour_of_day",
    "day_index",
    "orig_balance_ratio",
    "orig_insufficient_balance"
]
#Label
target_column = "isFraud"

model_df = features[
    feature_columns + [target_column]
].copy()

display(model_df.tail())

,step,type,amount,oldbalanceOrg,log_amount,hour_of_day,day_index,orig_balance_ratio,orig_insufficient_balance,isFraud
6362615,743,CASH_OUT,339682.13,339682.13,12.735768,22,30,0.999997,0,1
6362616,743,TRANSFER,6311409.28,6311409.28,15.657870,22,30,1.000000,0,1
6362617,743,CASH_OUT,6311409.28,6311409.28,15.657870,22,30,1.000000,0,1
6362618,743,TRANSFER,850002.52,850002.52,13.652996,22,30,0.999999,0,1
6362619,743,CASH_OUT,850002.52,850002.52,13.652996,22,30,0.999999,0,1


In [ ]:
OUTPUT_PATH = BASE_DIR / "data/processed/"
OUTPUT_FILE = OUTPUT_PATH / "paysim_featured.csv"

OUTPUT_PATH.mkdir(parents=True, exist_ok=False)

model_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Feature dataset:", model_df.shape)

PermissionError: [Errno 13] Permission denied: 'D:\\Study\\Fraud\\fraud-data-engineering\\data\\processed\\paysim_featured.csv'